In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ---------------------------------------------------------
# 1. Dataset Loading & Preprocessing
# ---------------------------------------------------------
# Load spam dataset (latin-1 encoding is standard for spam.csv)
df = pd.read_csv('spam.csv', encoding='latin-1')

# Select relevant columns and rename
df = df[['v1', 'v2']]
df.columns = ['label', 'text']

# Map target labels ('ham' -> 0, 'spam' -> 1)
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

# Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label_num'], test_size=0.2, random_state=42, stratify=df['label_num']
)

# ---------------------------------------------------------
# Part 1: CountVectorizer with stop_words
# ---------------------------------------------------------
# Feature extraction with CountVectorizer
count_vec = CountVectorizer(stop_words='english')
X_train_count = count_vec.fit_transform(X_train)
X_test_count = count_vec.transform(X_test)

# Train Multinomial Naive Bayes Model
mnb_count = MultinomialNB()
mnb_count.fit(X_train_count, y_train)

# Model Evaluation
y_pred_count = mnb_count.predict(X_test_count)
acc_count = accuracy_score(y_test, y_pred_count)

print("=" * 60)
print("PART 1: MULTINOMIAL NAIVE BAYES (CountVectorizer)")
print("=" * 60)
print(f"Accuracy: {acc_count:.4f} ({acc_count * 100:.2f}%)\n")
print("Classification Report:")
print(classification_report(y_test, y_pred_count, target_names=['ham', 'spam']))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_count))

# ---------------------------------------------------------
# Part 2: TF-IDF Vectorizer with stop_words
# ---------------------------------------------------------
# Feature extraction with TfidfVectorizer
tfidf_vec = TfidfVectorizer(stop_words='english')
X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_test_tfidf = tfidf_vec.transform(X_test)

# Train Multinomial Naive Bayes Model
mnb_tfidf = MultinomialNB()
mnb_tfidf.fit(X_train_tfidf, y_train)

# Model Evaluation
y_pred_tfidf = mnb_tfidf.predict(X_test_tfidf)
acc_tfidf = accuracy_score(y_test, y_pred_tfidf)

print("\n" + "=" * 60)
print("PART 2: MULTINOMIAL NAIVE BAYES (TF-IDF Vectorizer)")
print("=" * 60)
print(f"Accuracy: {acc_tfidf:.4f} ({acc_tfidf * 100:.2f}%)\n")
print("Classification Report:")
print(classification_report(y_test, y_pred_tfidf, target_names=['ham', 'spam']))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tfidf))

PART 1: MULTINOMIAL NAIVE BAYES (CountVectorizer)
Accuracy: 0.9839 (98.39%)

Classification Report:
              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       966
        spam       0.96      0.92      0.94       149

    accuracy                           0.98      1115
   macro avg       0.97      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115

Confusion Matrix:
[[960   6]
 [ 12 137]]

PART 2: MULTINOMIAL NAIVE BAYES (TF-IDF Vectorizer)
Accuracy: 0.9686 (96.86%)

Classification Report:
              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       966
        spam       1.00      0.77      0.87       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.97      1115

Confusion Matrix:
[[966   0]
 [ 35 114]]


- TF-IDF misses 35 spam messages (Spam Recall drops to 76.51%) because TF-IDF down-weights frequent terms, making prominent spam trigger words less influential.
- CountVectorizer only misses 12 spam messages (Spam Recall is 91.95%), achieving superior overall accuracy (98.39% vs. 96.86%).